# 🔬 rues-linker v0.7.5 — Medida **OBJETIVA** sobre datos reales (multi-fuente)

Para una medida objetiva del sistema completo hay que incluir **todas las fuentes**, no solo el
RUES: el RUES es el registro oficial y es **~100% CON_NIT**, así que por sí solo NO permite medir
**SIN_NIT** (el régimen difícil, donde viven los retos). SIN_NIT vive sobre todo en **CRM y
EXPORTACIONES**. Este notebook consolida las 4 fuentes, toma una muestra **balanceada por régimen**
(para que SIN_NIT sea medible), corre el motor, y mide **F1 real por régimen** tras etiquetar.

**Cómo usarlo:**
- **⚡ Prueba rápida:** `muestra_rapida = 5000` → minutos (Pasos 1-3).
- **🐢 Corrida completa / objetiva:** `muestra_rapida = None`; ajuste `n_muestra` (50 000 da una
  buena medida; suba a ~200 000 si quiere una corrida más larga y representativa).
- El esfuerzo de **etiquetado** lo fija `n_pares` (no `n_muestra`): aunque procese 200 000
  registros, solo etiqueta ~`n_pares` pares.
- Los Pasos 4-6 se saltan solos si aún no hay etiquetas. Al final, un **cronómetro** del proceso.

## Requisitos en su Drive
1. El paquete **`record_linkage`** (carpeta) o el tarball **`rues-linker-v0_7_5-FINAL.tar.gz`**.
2. La carpeta con las fuentes (`RUES_clasificado.zip`, `CRM_clasificado.zip`, etc.).

Para el paso de κ (sesión 2) el paquete debe traer `scripts/medir_kappa.py`; si no, use el tarball.

In [11]:
# ════════════════════════════════════════════════════════════════════
#  ⚙️  CONFIGURACIÓN — LA ÚNICA CELDA QUE USTED EDITA
# ════════════════════════════════════════════════════════════════════
from pathlib import Path

CONFIG = {
    # ── Paquete: carpeta de Drive O tarball (.tar.gz) ──
    'ruta_paquete': '/content/drive/MyDrive/ProColombia/record_linkage',

    # ── Carpeta donde están las fuentes ──
    'dir_datos': '/content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-02-14',

    # ── Fuentes a consolidar (NIT + RAZON_SOCIAL). Quite/agregue según tenga.
    #    RUES es ~todo CON_NIT; SIN_NIT viene sobre todo de CRM/EXPORTACIONES. ──
    'fuentes': [
        {'nombre': 'RUES',           'archivo': 'RUES_clasificado.zip',           'sep': '|', 'encoding': 'latin-1'},
        {'nombre': 'EXPORTACIONES',  'archivo': 'EXPORTACIONES_clasificado.zip',  'sep': ',', 'encoding': 'utf-8'},
        {'nombre': 'CRM',            'archivo': 'CRM_clasificado.zip',            'sep': ',', 'encoding': 'utf-8'},
        {'nombre': 'SUPERSOCIEDADES','archivo': 'SUPERSOCIEDADES_clasificado.zip','sep': ',', 'encoding': 'utf-8'},
    ],

    # ── Columnas ──
    'col_nit':  'NIT',
    'col_name': 'RAZON_SOCIAL',

    # ── Carpeta de salida en Drive (se crea sola tras montar) ──
    'out_dir':  '/content/drive/MyDrive/ProColombia/Deduplicación/prueba_rues_linker',

    # ══ ⚡ PRUEBA RÁPIDA vs 🐢 COMPLETA ════════════════════════════════
    # Número pequeño (p.ej. 5000) = lee solo esas filas POR FUENTE → minutos.
    # None = COMPLETA (lee todo y muestrea n_muestra).
    'muestra_rapida': None,

    'n_muestra':         50000,  # tamaño de la muestra a procesar (COMPLETA)
    'balancear_regimen': True,   # ~mitad CON_NIT / mitad SIN_NIT, para PODER medir SIN_NIT
    'n_pares':           500,    # nº de pares a etiquetar (define el esfuerzo humano)

    # ── Otros ──
    'mode':           'BALANCEADO',
    'seed':           42,
    'stats_clusters': True,
}

# ──────── derivados (NO editar) ────────
if CONFIG['muestra_rapida'] is not None:
    N_LEER = int(CONFIG['muestra_rapida']); N_MUESTRA = N_LEER
    N_PARES = min(int(CONFIG['n_pares']), max(60, N_LEER // 12))
    print(f"⚡ PRUEBA RÁPIDA — lee {N_LEER:,} filas/fuente · muestra {N_MUESTRA:,} · {N_PARES} pares")
else:
    N_LEER, N_MUESTRA, N_PARES = None, int(CONFIG['n_muestra']), int(CONFIG['n_pares'])
    print(f"🐢 CORRIDA COMPLETA — muestra {N_MUESTRA:,} · {N_PARES} pares (puede tardar)")

OUT = Path(CONFIG['out_dir'])   # la carpeta se crea en SETUP, DESPUÉS de montar Drive
MUESTRA  = OUT / 'muestra.parquet'
PARES    = OUT / 'pares_para_etiquetar.csv'
ETIQ_A   = OUT / 'etiquetas_A.csv'     # <- la llena el anotador A
ETIQ_B   = OUT / 'etiquetas_B.csv'     # <- la llena el anotador B
CONSENSO = OUT / 'pares_consenso.csv'
KAPPA_MD = OUT / 'kappa_reporte.md'
REPORTE  = OUT / 'resultado_real.md'
ERRORES  = OUT / 'errores_reales.csv'
print('Artefactos →', OUT)


🐢 CORRIDA COMPLETA — muestra 50,000 · 500 pares (puede tardar)
Artefactos → /content/drive/MyDrive/ProColombia/Deduplicación/prueba_rues_linker


In [12]:
# ════════════════════════════════════════════════════════════════════
#  SETUP — ejecutar UNA vez por sesión (no se edita)
# ════════════════════════════════════════════════════════════════════
import os, sys, subprocess, time
from pathlib import Path
from google.colab import drive

if not os.path.ismount('/content/drive'):
    try:
        drive.mount('/content/drive')
    except ValueError:
        drive.mount('/content/drive', force_remount=True)
else:
    print('ℹ️  Drive ya estaba montado')

PKG = Path(CONFIG['ruta_paquete'])
assert PKG.exists(), f"No encuentro el paquete en {PKG}"
if str(PKG).endswith(('.tar.gz', '.tgz')):
    subprocess.run(['tar', '-xzf', str(PKG), '-C', '/content'], check=True)
    cands = [p for p in Path('/content').glob('rues-linker*') if p.is_dir()]
    assert cands, "El tarball no se extrajo como esperaba."
    REPO = cands[0]
else:
    REPO = PKG
assert (REPO / 'scripts').exists(), f"No hay carpeta scripts/ en {REPO}"
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)

import importlib, site
importlib.reload(site)
for _cand in [REPO / 'src', REPO]:
    if (_cand / 'record_linkage' / '__init__.py').exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand))
importlib.invalidate_caches()
import record_linkage
print('rues-linker', record_linkage.__version__, '· REPO =', REPO)

OUT.mkdir(parents=True, exist_ok=True)
print('Artefactos →', OUT)

_necesarios = ['generar_pares_para_etiquetar.py', 'medir_con_ground_truth.py']
_faltan = [s for s in _necesarios if not (REPO / 'scripts' / s).exists()]
assert not _faltan, f"Al paquete le faltan scripts: {_faltan}. Use el tarball que le entregué."
TIENE_KAPPA = (REPO / 'scripts' / 'medir_kappa.py').exists()
print('medir_kappa.py disponible:', TIENE_KAPPA)

def run_script(name, args, tail=4000):
    cmd = [sys.executable, str(REPO / 'scripts' / name)] + [str(a) for a in args]
    print('▶', name, ' '.join(str(a) for a in args), '\n')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout[-tail:])
    if r.returncode != 0:
        print('--- STDERR ---'); print(r.stderr[-3000:])
        raise SystemExit(f'❌ {name} falló (código {r.returncode})')
    return r

TIEMPOS = {}
def _fin(nombre, t0):
    TIEMPOS[nombre] = round(time.time() - t0, 1)
    print(f"\n⏱️  {nombre}: {TIEMPOS[nombre]} s")


ℹ️  Drive ya estaba montado
rues-linker 0.7.5 · REPO = /content/drive/MyDrive/ProColombia/record_linkage
Artefactos → /content/drive/MyDrive/ProColombia/Deduplicación/prueba_rues_linker
medir_kappa.py disponible: True


## Paso 1 — Consolidar las fuentes y muestrear (balanceado por régimen)
Lee cada fuente (solo NIT + RAZON_SOCIAL, para no agotar memoria), las concatena, y toma la muestra.
Con `balancear_regimen=True` la muestra trae ~mitad CON_NIT y ~mitad SIN_NIT, para poder medir SIN_NIT.

In [13]:
import time; _t0 = time.time()
import warnings; warnings.filterwarnings('ignore')
import pandas as pd

nit, name = CONFIG['col_nit'], CONFIG['col_name']
dir_datos = Path(CONFIG['dir_datos'])
partes = []
for f in CONFIG['fuentes']:
    ruta = dir_datos / f['archivo']
    if not ruta.exists():
        print(f"⚠️  Falta {f['nombre']}: {ruta} → se omite"); continue
    suf = ruta.suffix.lower()
    comp = 'zip' if suf == '.zip' else ('gzip' if suf == '.gz' else None)
    d = pd.read_csv(ruta, sep=f['sep'], encoding=f['encoding'], compression=comp,
                    usecols=lambda c: c in (nit, name), dtype=str, nrows=N_LEER, on_bad_lines='skip')
    if not {nit, name}.issubset(d.columns):
        print(f"⚠️  {f['nombre']} no tiene {nit}/{name} (tiene {list(d.columns)}) → se omite"); continue
    d = d[[nit, name]].copy(); d['SRC'] = f['nombre']
    partes.append(d)
    print(f"  {f['nombre']:<16}{len(d):>10,} filas")
assert partes, "No se cargó ninguna fuente. Revise dir_datos y los nombres de archivo."

df = pd.concat(partes, ignore_index=True)
con = df[nit].notna() & (df[nit].astype(str).str.strip().replace('nan', '') != '')
print(f"\nConsolidado: {len(df):,} filas | CON_NIT {con.sum():,} ({con.mean()*100:.1f}%) | SIN_NIT {(~con).sum():,} ({(~con).mean()*100:.1f}%)")
if (~con).sum() == 0:
    print("⚠️  No hay registros SIN_NIT en lo cargado → NO se podrá medir el régimen SIN_NIT.")

if CONFIG['balancear_regimen'] and (~con).sum() > 0:
    half = N_MUESTRA // 2
    dsin, dcon = df[~con], df[con]
    n_sin = min(len(dsin), half); n_con = min(len(dcon), N_MUESTRA - n_sin)
    muestra = pd.concat([dcon.sample(n_con, random_state=CONFIG['seed']) if len(dcon) > n_con else dcon,
                         dsin.sample(n_sin, random_state=CONFIG['seed']) if len(dsin) > n_sin else dsin])
    print(f"Muestra BALANCEADA por régimen: {n_con:,} CON_NIT + {n_sin:,} SIN_NIT")
else:
    muestra = df.sample(N_MUESTRA, random_state=CONFIG['seed']) if len(df) > N_MUESTRA else df

muestra = muestra.sample(frac=1, random_state=CONFIG['seed']).reset_index(drop=True)
muestra.to_parquet(MUESTRA, index=False)
mc = muestra[nit].notna() & (muestra[nit].astype(str).str.strip().replace('nan', '') != '')
print(f"Muestra final: {len(muestra):,} filas | CON_NIT {mc.sum():,} | SIN_NIT {(~mc).sum():,} → {MUESTRA}")
display(muestra[[nit, name, 'SRC']].head(5))
_fin('Paso 1 (consolidar+muestra)', _t0)


  RUES             1,919,738 filas
  EXPORTACIONES       17,345 filas
  CRM                 18,649 filas
  SUPERSOCIEDADES     10,000 filas

Consolidado: 1,965,732 filas | CON_NIT 1,965,732 (100.0%) | SIN_NIT 0 (0.0%)
⚠️  No hay registros SIN_NIT en lo cargado → NO se podrá medir el régimen SIN_NIT.
Muestra final: 50,000 filas | CON_NIT 50,000 | SIN_NIT 0 → /content/drive/MyDrive/ProColombia/Deduplicación/prueba_rues_linker/muestra.parquet


,NIT,RAZON_SOCIAL,SRC
0,901276424,HABITAT PUBLICITARIA S.A.S,RUES
1,901356253,PULPAS ZANJOSE S.A.S.,RUES
2,CC12902672,SINISTERRA BASTIDAS BERNARDO,RUES
3,CC1007145897,QUILINDO DELGADO JUAN JOSE,RUES
4,CC1049615462,ROJAS ALVAREZ YENIFER ELESVY,RUES



⏱️  Paso 1 (consolidar+muestra): 9.8 s


## Paso 2 — Correr el motor: cuántos duplicados encuentra (resultado rápido)
Es **cantidad**, no calidad. La calidad (F1) se mide en los Pasos 4-6 (tras etiquetar).

In [14]:
import time; _t0 = time.time()
if CONFIG['stats_clusters']:
    import warnings; warnings.filterwarnings('ignore')
    import pandas as pd
    from record_linkage.deduplication.auto import deduplicate_auto

    m = pd.read_parquet(MUESTRA)
    corr, stats = deduplicate_auto(m, col_nit=CONFIG['col_nit'], col_name=CONFIG['col_name'], mode=CONFIG['mode'])
    nitc = CONFIG['col_nit']
    n = len(corr); ng = corr['ID_GRUPO'].nunique()
    con = corr[nitc].notna() & (corr[nitc].astype(str).str.strip().replace('nan', '') != '')
    print(f"Registros: {n:,}  →  Entidades únicas: {ng:,}   |   Reducción: {(1 - ng / n) * 100:.1f}%")
    print(f"CON_NIT: {con.sum():,} regs → {corr.loc[con, 'ID_GRUPO'].nunique():,} grupos")
    print(f"SIN_NIT: {(~con).sum():,} regs → {corr.loc[~con, 'ID_GRUPO'].nunique():,} grupos")
    print('\nGrupos más grandes (tamaño):')
    print(corr['ID_GRUPO'].value_counts().head(5).to_string())
    print('\n⚠️ CANTIDAD, no calidad. El F1 real = Pasos 4-6.')
else:
    print('Paso 2 omitido (stats_clusters=False).')
_fin('Paso 2 (motor)', _t0)


2026-05-31 09:31:24 | deduplicate_auto          | INFO     | deduplicate_auto: 50000 CON_NIT, 0 SIN_NIT
2026-05-31 09:31:24 | deduplicate_auto          | INFO     | Dataset homogéneo CON_NIT → deduplication_standard


2026-05-31 14:31:24 - deduplicate_unified - INFO - Iniciando deduplicación unificada: 50,000 registros


2026-05-31 09:31:24 | deduplicate_unified       | INFO     | Iniciando deduplicación unificada: 50,000 registros


2026-05-31 14:31:24 - dedup_preparation - INFO - Procesando nombres con EnhancedTextProcessor...


2026-05-31 09:31:24 | dedup_preparation         | INFO     | Procesando nombres con EnhancedTextProcessor...


...

2026-05-31 09:31:25 | dedup_preparation         | INFO     | Removiendo top-20 palabras del corpus...


2026-05-31 14:31:25 - dedup_preparation - INFO - Procesando NITs con AdvancedNitProcessor...


2026-05-31 09:31:25 | dedup_preparation         | INFO     | Procesando NITs con AdvancedNitProcessor...
2026-05-31 09:31:26 | record_linkage.pipeline.linkage_pipeline | INFO     | ▶️  Iniciando: Pipeline completo
2026-05-31 09:31:26 | record_linkage.pipeline.linkage_pipeline | INFO     |     💾 Estado Memoria Inicial: 0.9GB usados por el proceso (14.2% del sistema)
2026-05-31 09:31:26 | record_linkage.pipeline.linkage_pipeline | INFO     | ▶️  Iniciando: Fase 1: Carga y Validación
2026-05-31 09:31:26 | record_linkage.pipeline.linkage_pipeline | INFO     |     💾 Estado Memoria Inicial: 0.9GB usados por el proceso (14.2% del sistema)
2026-05-31 09:31:26 | record_linkage.pipeline.linkage_pipeline | INFO     | ✅ Completado: Fase 1: Carga y Validación en 0.29s
2026-05-31 09:31:26 | record_linkage.pipeline.linkage_pipeline | INFO     |    📈 Estado Memoria Final: 0.9GB usados (14.4% del sistema) | Consumo en fase: +24.9 MB | Salud del Sistema: 🟢 NORMAL
2026-05-31 09:31:26 | record_linkage.pip

2026-05-31 14:31:55 - deduplicate_unified - INFO - Deduplicación completada en 31.0s. Archivos guardados: correlativa_unificada.parquet, conexiones_no_triviales.parquet


2026-05-31 09:31:55 | deduplicate_unified       | INFO     | Deduplicación completada en 31.0s. Archivos guardados: correlativa_unificada.parquet, conexiones_no_triviales.parquet
Registros: 50,000  →  Entidades únicas: 49,667   |   Reducción: 0.7%
CON_NIT: 50,000 regs → 49,667 grupos
SIN_NIT: 0 regs → 0 grupos

Grupos más grandes (tamaño):
ID_GRUPO
12139    4
255      4
24947    4
4084     4
7071     4

⚠️ CANTIDAD, no calidad. El F1 real = Pasos 4-6.

⏱️  Paso 2 (motor): 31.1 s


## Paso 3 — Generar los pares para etiquetar (estratificados por score)

In [15]:
import time; _t0 = time.time()
run_script('generar_pares_para_etiquetar.py', [
    '--input', MUESTRA, '--col-nit', CONFIG['col_nit'], '--col-name', CONFIG['col_name'],
    '--salida', PARES, '--n-pares', N_PARES, '--mode', CONFIG['mode'], '--seed', CONFIG['seed'],
])
import pandas as pd
pares = pd.read_csv(PARES)
print(f"\n✅ {len(pares):,} pares → {PARES}")
print('Columnas:', list(pares.columns)); display(pares.head(3))
if 'ESTRATO' in pares.columns:
    print('\n📊 Pares por estrato:'); print(pares['ESTRATO'].value_counts().to_string())
_fin('Paso 3 (pares)', _t0)


▶ generar_pares_para_etiquetar.py --input /content/drive/MyDrive/ProColombia/Deduplicación/prueba_rues_linker/muestra.parquet --col-nit NIT --col-name RAZON_SOCIAL --salida /content/drive/MyDrive/ProColombia/Deduplicación/prueba_rues_linker/pares_para_etiquetar.csv --n-pares 500 --mode BALANCEADO --seed 42 

📥 Leyendo /content/drive/MyDrive/ProColombia/Deduplicación/prueba_rues_linker/muestra.parquet...
   50,000 registros, 49,995 NITs únicos
⚙️  Preparando dataset y bloqueando candidatos...
   30,403 pares candidatos generados
🎯 Scoreando pares para estratificar...
   30,403 pares scoreados
📊 Muestreando ~500 pares estratificados...
   337 pares seleccionados

Distribución por estrato:
ESTRATO
alta_confianza                 100
potencial_fn                   100
nombre_alto_nit_distinto        94
frontera                        38
nit_identico_nombre_disimil      5

📝 Construyendo plantilla de etiquetado...

✅ 337 pares escritos en: /content/drive/MyDrive/ProColombia/Deduplicación/pru

,par_id,nit_a,razon_a,nit_b,razon_b,score_sistema,name_sim,nit_dist,ESTRATO,MISMO_GRUPO,CASO_FRONTERA,NOTAS
0,P00001,890306462,FUNDACION SMURFIT WESTROCK COLOMBIA,8903064623,SMURFIT KAPPA-WESTROCK,0.955,1.0,0,alta_confianza,NaN,NaN,NaN
1,P00002,900324294,ZONA DE ACTIVIDAD LOGISTICA S A S,900023091,ACTIVIDAD INMOBILIARIA S.A.S.,0.920,1.0,4,alta_confianza,NaN,NaN,NaN
2,P00003,900567095,O.A.C. CONSTRUCTORES SAS,900557572,DISTRIBUCIONES Y SERVICIOS INTEGRALES C&S S.A.S,0.900,1.0,5,alta_confianza,NaN,NaN,NaN



📊 Pares por estrato:
ESTRATO
alta_confianza                 100
potencial_fn                   100
nombre_alto_nit_distinto        94
frontera                        38
nit_identico_nombre_disimil      5

⏱️  Paso 3 (pares): 7.8 s


---
# ⏸️ PARE AQUÍ para etiquetar

Si corre los Pasos 4-6 ahora sin etiquetas, **se saltan solos** (no falla).

**Para la medición real de F1:**
1. Descargue `pares_para_etiquetar.csv`. Dos personas, por separado, llenan `MISMO_GRUPO`
   con `True`/`False` (mismo conjunto o solape ≥ 20%).
2. Guárdenlas en `out_dir` como `etiquetas_A.csv` y `etiquetas_B.csv` (conserven `par_id`).
3. Reabra el notebook, corra CONFIG + SETUP, y siga desde el Paso 4.

Protocolo: `docs/PROTOCOLO_GROUND_TRUTH.md`.

---

## Paso 4 — Acuerdo entre etiquetadores (Cohen's κ) — **compuerta κ ≥ 0.80**

In [16]:
import os, time; _t0 = time.time()
if not (os.path.exists(ETIQ_A) and os.path.exists(ETIQ_B)):
    print("⏸️  Aún no hay etiquetas. Faltan:",
          ", ".join(f.name for f in (ETIQ_A, ETIQ_B) if not os.path.exists(f)))
    print("    Complete el etiquetado (bloque '⏸️ PARE AQUÍ') y vuelva a este paso.")
elif not TIENE_KAPPA:
    print("⚠️  Este paquete no tiene scripts/medir_kappa.py. Use rues-linker-v0_7_5-FINAL.tar.gz.")
else:
    import re
    r = run_script('medir_kappa.py', [
        '--labeler-a', ETIQ_A, '--labeler-b', ETIQ_B,
        '--col-id', 'par_id', '--col-label', 'MISMO_GRUPO', '--report', KAPPA_MD,
    ])
    m = re.search(r'(?:kappa|κ)[^0-9\-]*(-?\d\.\d+)', r.stdout, re.I)
    if m:
        k = float(m.group(1)); print(f"\nCohen's κ = {k:.3f}")
        print("✅ COMPUERTA OK (κ ≥ 0.80): etiquetado confiable. Continúe." if k >= 0.80
              else "❌ NO PASA (κ < 0.80): NO mida F1 aún; resuelvan desacuerdos / reescriban el manual.")
    else:
        print("\n(Lea κ arriba o en", KAPPA_MD, ")")
_fin('Paso 4 (κ)', _t0)


⏸️  Aún no hay etiquetas. Faltan: etiquetas_A.csv, etiquetas_B.csv
    Complete el etiquetado (bloque '⏸️ PARE AQUÍ') y vuelva a este paso.

⏱️  Paso 4 (κ): 0.0 s


## Paso 5 — Construir el consenso (pares donde ambos coinciden)

In [17]:
import os, time; _t0 = time.time()
if not (os.path.exists(ETIQ_A) and os.path.exists(ETIQ_B)):
    print("⏸️  Aún no hay etiquetas. Complete el etiquetado y vuelva a este paso.")
else:
    import pandas as pd
    def to_bool(s):
        mp = {'si': True, 'sí': True, '1': True, 'true': True, 'verdadero': True, 'yes': True, 'match': True,
              'no': False, '0': False, 'false': False, 'falso': False, 'no_match': False}
        return s.astype(str).str.strip().str.lower().map(mp)
    A = pd.read_csv(ETIQ_A); B = pd.read_csv(ETIQ_B)
    A['_la'] = to_bool(A['MISMO_GRUPO']); B['_lb'] = to_bool(B['MISMO_GRUPO'])
    both = A.merge(B[['par_id', '_lb']], on='par_id').dropna(subset=['_la', '_lb'])
    agreed = both[both['_la'] == both['_lb']].copy()
    disag = both[both['_la'] != both['_lb']]
    cons = agreed.drop(columns=['_lb'])
    cons['MISMO_GRUPO'] = agreed['_la'].map({True: 'True', False: 'False'})
    cons = cons.drop(columns=['_la'])
    cons.to_csv(CONSENSO, index=False)
    print(f"Pares etiquetados por ambos: {len(both):,}")
    print(f"  De acuerdo (→ consenso):   {len(agreed):,}")
    print(f"  En desacuerdo (excluidos): {len(disag):,}")
    print(f"✅ Consenso → {CONSENSO}")
    if len(disag):
        print("\n⚠️ Para rigor, resuelvan los desacuerdos en sesión y reincorpórenlos antes de re-medir.")
_fin('Paso 5 (consenso)', _t0)


⏸️  Aún no hay etiquetas. Complete el etiquetado y vuelva a este paso.

⏱️  Paso 5 (consenso): 0.0 s


## Paso 6 — F1/Precision/Recall real por régimen (CON_NIT/SIN_NIT) y estrato

In [18]:
import os, time; _t0 = time.time()
if not os.path.exists(CONSENSO):
    print("⏸️  Aún no hay consenso. Corra el Paso 5 (tras etiquetar) y vuelva aquí.")
else:
    run_script('medir_con_ground_truth.py', [
        '--input', MUESTRA, '--pares-etiquetados', CONSENSO,
        '--col-nit', CONFIG['col_nit'], '--col-name', CONFIG['col_name'],
        '--mode', CONFIG['mode'], '--reporte', REPORTE, '--errores-csv', ERRORES,
    ])
    from pathlib import Path
    print('\n================ REPORTE (real) ================\n')
    print(Path(REPORTE).read_text()[:6000] if Path(REPORTE).exists() else 'No se generó el reporte.')
_fin('Paso 6 (F1)', _t0)


⏸️  Aún no hay consenso. Corra el Paso 5 (tras etiquetar) y vuelva aquí.

⏱️  Paso 6 (F1): 0.0 s


## Decisión — con el F1 **real** (mire sobre todo SIN_NIT)

| F1 real | Decisión |
|---|---|
| **≥ 0.90** | El motor sirve. Endurecer y publicar con cifras reales. |
| **0.80 – 0.90** | Atacar los 5 modos de falla de `errores_reales.csv` (recalibración / Fase 3). |
| **< 0.80** | Problema estructural: DuckDB/Splink + enriquecer campos dejan de ser opcionales. |

El **F1 SIN_NIT** es el que decide: si es bajo, no es el código — es la falta de señal en la razón
social sola. La salida es enriquecer campos (dirección, fechas, CIIU), que el motor ya soporta.

In [19]:
import os, re, time; _t0 = time.time()
from pathlib import Path
txt = Path(REPORTE).read_text() if os.path.exists(REPORTE) else ''
if not txt:
    print("⏸️  Aún no hay medición. Complete los Pasos 4-6 (tras etiquetar) y vuelva aquí.")
else:
    m = re.search(r'F1[^0-9]*?(\d\.\d+)', txt)
    f1 = float(m.group(1)) if m else None
    if f1 is not None:
        print(f"F1 real (global) ≈ {f1:.3f}")
        if f1 >= 0.90:   print("→ ≥0.90: el motor sirve.")
        elif f1 >= 0.80: print("→ 0.80–0.90: revise errores_reales.csv; recalibración / Fase 3.")
        else:            print("→ <0.80: problema estructural (Splink/DuckDB + enriquecer campos).")
    else:
        print("Lea el F1 en el reporte de arriba.")
    print("\n📋 Pegue κ y los F1 (global, por régimen, por estrato) en docs/MEDICION_REAL.md.")
_fin('Decisión', _t0)


⏸️  Aún no hay medición. Complete los Pasos 4-6 (tras etiquetar) y vuelva aquí.

⏱️  Decisión: 0.0 s


## ⏱️ Tiempo total del proceso

In [20]:
def mostrar_tiempos(tiempos):
    """Imprime el tiempo de cada paso y el TOTAL del proceso (sin contar el etiquetado manual)."""
    print("=" * 56)
    print("  ⏱️  TIEMPO DEL PROCESO  (no incluye el etiquetado manual)")
    print("=" * 56)
    total = 0.0
    for nombre, seg in tiempos.items():
        print(f"  {nombre:<32} {seg:>9.1f} s")
        total += seg
    mins, secs = divmod(int(round(total)), 60)
    print("-" * 56)
    print(f"  {'TOTAL':<32} {total:>9.1f} s   ≈  {mins} min {secs:02d} s")
    print("=" * 56)

mostrar_tiempos(TIEMPOS)


  ⏱️  TIEMPO DEL PROCESO  (no incluye el etiquetado manual)
  Paso 1 (consolidar+muestra)            9.8 s
  Paso 2 (motor)                        31.1 s
  Paso 3 (pares)                         7.8 s
  Paso 4 (κ)                             0.0 s
  Paso 5 (consenso)                      0.0 s
  Paso 6 (F1)                            0.0 s
  Decisión                               0.0 s
--------------------------------------------------------
  TOTAL                                 48.7 s   ≈  0 min 49 s


# FIN